## Interior point method

In [2]:
import numpy as np
import pandas as pd
import copy as copy
import scipy
import scipy.io
import time
import os
from scipy.linalg import solve, LinAlgWarning
import warnings
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from matplotlib.animation import FuncAnimation, PillowWriter
import openpyxl
import xlsxwriter

from inpoint_methods import paso_intpoint, loadProblem, intpoint, intpointR, highlight_greaterthan #,intpointR_mask

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [3]:
def solve_catch(K,ld):
    with warnings.catch_warnings(record=True) as warneo:
        warnings.simplefilter("always", LinAlgWarning)
        # Solve the linear system
        w_vector = scipy.linalg.solve(K, ld)
        
        # Check if any warning was triggered
        if any(issubclass(warn.category, LinAlgWarning) for warn in warneo):
            # Print the warning message for all warnings captured
            for warn in warneo:
                print(f"Warning: {warn.message}")
    return w_vector

def highlight_df(mu, z, Q, k, tau, highlighted_df, mudf, epsilon=1e-6, comp_tol=1e-5):
    # Previous μ from mudf
    prev_mu = mudf.loc[k-1].values
    
    # Only consider indices where μ*z <= comp_tol
    mask = mu * z <= comp_tol
    
    # Highlight if μ did not increase significantly and μ*z is small enough
    highlighted_rows = [i for i in range(len(mu)) if mask[i] and mu[i] <= prev_mu[i] + epsilon]
    
    # Build row of 1s and 0s
    highlighted_row = [1 if i in highlighted_rows else 0 for i in range(Q.shape[0])]
    
    # Store in DataFrame
    highlighted_df.loc[k] = highlighted_row
    return highlighted_df

def display_highlights(highlighted_df,regression_df):
    # Take last 3 iterations
    last_3_iterations = highlighted_df.tail(3)
    
    # Columns consistently highlighted (1 in all last 3 iterations)
    highlighted_columns = last_3_iterations.columns[(last_3_iterations == 1).all(axis=0)].tolist()
    highlighted_columns = [int(x) for x in highlighted_columns]  # ensure ints
    print("Consistently highlighted in last 3 iterations:", highlighted_columns)
    print("Consistently highlighted in last 3 iterations:", highlighted_columns)
    print("How many in percentage of mu's dimension? ", (len(highlighted_columns)/p)*100,"%")
    #print("How many?=", len(highlighted_columns))
    
    # Columns that were highlighted in previous iteration but regressed this iteration (1 → 0)
    regressed_columns = []
    if len(highlighted_df) >= 2:
        prev_iter = highlighted_df.iloc[-2]
        last_iter = highlighted_df.iloc[-1]
        regressed_columns = prev_iter.index[(prev_iter == 1) & (last_iter == 0)].tolist()
        regressed_columns = [int(x) for x in regressed_columns]
        if regressed_columns:
            print(f"WARNING: Iter {highlighted_df.index[-1]} regressed columns (1 -> 0):", regressed_columns)

            highlighted_df_regressed = highlighted_df.tail(10)[regressed_columns]
            display(highlighted_df_regressed)
        iter_idx = highlighted_df.index[-1]
        if regression_df is not None:
                row = pd.Series(0, index=highlighted_df.columns)
                row[regressed_columns] = 1
                regression_df.loc[iter_idx] = row
    
    return highlighted_columns, regressed_columns


def progress_summary_df_clean_simple(metrics_before, metrics_after):
    """
    Build a progress summary DataFrame given 'before' and 'after' metrics.
    metrics_before/after: dict with keys as metric names and values as numbers.
    """

    summary_data = {
        "Metric": [],
        "Before": [],
        "After": [],
        "Improved?": []
    }

    for metric in metrics_before:
        before = metrics_before[metric]
        after = metrics_after[metric]

        summary_data["Metric"].append(metric)
        summary_data["Before"].append(before)
        summary_data["After"].append(after)

        # Smaller is better
        improved = after < before
        summary_data["Improved?"].append(improved)

    summary_df = pd.DataFrame(summary_data)
    return summary_df

def build_reduced_system(Q, AT, FT, U, A, F, Z, mu, x, lamda, c, b, d, tau, highlighted_columns):
    """
    Builds the full KKT system M and a reduced system M1 by eliminating
    rows/columns corresponding to highlighted indices.
    
    Returns:
        M      : full KKT system
        M1     : reduced KKT system (square)
        U1     : filtered diagonal of mu for reduced system
        ld1    : reduced RHS vector
    """
    # Dimensions
    n = Q.shape[0]
    m = A.shape[0]
    p = U.shape[0]

    # ────────── Build full system ──────────
    r1 = np.hstack((Q, AT, -FT @ U))
    r2 = np.hstack((A, np.zeros((m, m + p))))
    r3 = np.hstack((-U @ F, np.zeros((p, m)), -Z @ U))

    M = np.vstack((r1, r2, r3))   # full KKT system

    # ────────── Filter mu ──────────
    active_indices = [i for i in range(p) if i not in highlighted_columns]
    mu_filtered = mu[active_indices]
    U1 = np.diag(mu_filtered)

    # ────────── Build reduced system ──────────
    rows_to_remove = [i + (n + m) for i in highlighted_columns]  # only the μ rows
    M1 = np.delete(M, rows_to_remove, axis=0)
    M1 = np.delete(M1, rows_to_remove, axis=1)

    if M1.shape[0] != M1.shape[1]:
        raise ValueError("M1 is not square! Check highlighted indices.")

    # ────────── Build reduced RHS vector ──────────
    ld_full = np.concatenate((
        Q @ x + AT @ lamda - FT @ mu + c,   # dual residual
        A @ x - b,                          # primal residual
        U @ (d - F @ x) + tau               # complementarity row
    ))

    ld1 = np.delete(ld_full, rows_to_remove, axis=0)

    print(f"Deleted {len(rows_to_remove)} rows/columns. M1 shape: {M1.shape}")

    return M, M1, U1, ld1

def load_lp_problem(mat_file: str):
    """
    Loads a linear/quadratic problem from a .mat file and initializes
    the problem matrices for the interior-point algorithm.
    
    Returns:
        Q : ndarray    Quadratic matrix (identity by default)
        c : ndarray    Linear term vector
        A : ndarray    Equality constraint matrix
        b : ndarray    Equality constraint RHS
        F : ndarray    Inequality constraint matrix (identity by default)
        d : ndarray    Inequality constraint RHS (zeros)
        H : dict       Raw data loaded from the .mat file
    """
    print(f"Loading problem from: {mat_file}")
    H = loadProblem(f"mat_files/{mat_file}")

    # Quadratic term: identity (can be changed if needed)
    Q = np.eye(H['c'].shape[0])

    # Linear term
    c = H['c'].ravel()  # flatten in case it's a column vector

    # Equality constraints
    A = H['AE']
    b = H['bE'].ravel()  # flatten in case it's a column vector

    # Compute percentage of zeros in b
    num_zeros_b = np.sum(b == 0)
    pct_zeros_b = 100 * num_zeros_b / len(b)
    
    # Inequality constraints (x >= 0 by default)
    F = np.eye(H['c'].shape[0])
    d = np.zeros(H['c'].shape[0])

    print(f"Problem loaded. n={Q.shape[0]}, m={A.shape[0]}, p={F.shape[0]}")
    #print(f"Equality RHS b: {b}")
    print(f"Number of zeros in b: {num_zeros_b} ({pct_zeros_b:.2f}%)")

    return Q, c, A, b, F, d, H


def summary_excel(summary_df, mat_files, n, m, p, b, regressed_df=None, folder="progress_summary"):
    """
    Add problem info to summary_df, reorder columns, and save to Excel.
    
    Args:
        summary_df (pd.DataFrame): DataFrame with progress metrics.
        mat_files (str): Name of the problem file.
        n, m, p (int): Dimensions of x, equality, and inequality constraints.
        b (np.ndarray): Equality constraint RHS.
        regressed_df (pd.DataFrame, optional): DataFrame with regressions, if any.
        folder (str): Folder to save the Excel file (default "progress_summary").
    """
    import os
    os.makedirs(folder, exist_ok=True)

    # Add problem info
    summary_df['Problem'] = mat_files
    summary_df['dim_x'] = n
    summary_df['dim_A'] = m
    summary_df['dim_F'] = p
    summary_df['||b||_inf'] = np.linalg.norm(b, np.inf)
    summary_df['num_zeros_in_b'] = np.sum(b == 0)

    # Reorder columns
    cols_order = ['Problem', 'dim_x', 'dim_A', 'dim_F', '||b||_inf', 'num_zeros_in_b',
                  'Metric', 'Before', 'After']
    summary_df = summary_df[cols_order]

    # Save to Excel
    excel_path = f"{folder}/progress_summary_{mat_files}.xlsx"
    with pd.ExcelWriter(excel_path, engine='xlsxwriter') as writer:
        summary_df.to_excel(writer, sheet_name='Summary', index=False)
        if regressed_df is not None and not regressed_df.empty:
            regressed_df.to_excel(writer, sheet_name='Regressions', index=False)

    print(f"Progress summary with regressions saved for {mat_files} at {excel_path}")
    return summary_df

Load problem

In [4]:
#mat_files = 'lp_kb2.mat'     # b = 0
#mat_files = 'lp_afiro.mat'   # bmax = 500, and no mu tends to zero  and no condition problem
mat_files = 'lp_blend.mat'    # bmax = 26.32, and no mu tends to zero.
#mat_files = 'lp_fit1d.mat'   # b = 0
#mat_files = 'lp_fit1p.mat'   # bmax = 216, and no mu tends to zero, condition 10^12
#mat_files = 'lp_grow15.mat'  # b = 0, condition 10^6
#mat_files = 'lp_grow22.mat'  # b = 0, condition 1.4e7
#mat_files = 'lp_grow7.mat'   # b = 0, condition 1.2e7

print(mat_files)

H = loadProblem(f"mat_files/{mat_files}")

# Define the quadratic matrix Q
Q = np.eye(H['c'].shape[0])

# Define the linear term vector c
c = H['c']

# Define the equality constraint matrix A and vector b
A = H['AE']
b = H['bE']
#b = np.zeros( A.shape[0] )

# Define an inequality constraint: x1 >= -10
#F = np.zeros([H['c'].shape[0],H['c'].shape[0]])
F = np.eye(H['c'].shape[0])
d = np.zeros([H['c'].shape[0],1])
d = d.ravel()  # Flattens the vector to a 1D vector
print(b)

lp_blend.mat
 Norma infinita de b:  26.32
[ 0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
  0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
  0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
  0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
  0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
  0.    0.    0.    0.   23.26  5.25 26.32 21.05 13.45  2.58 10.   10.
  0.    0.  ]


In [5]:
# Sistema Completo
#x, lamda, mu, z, k = intpoint(Q, A, F, c, b, d)
# Sistema Reducido
#x, lamda, mu, z, k = intpointR(Q, A, F, c, b, d)

# Método de puntos interiores con heurística (¿?)

In [ ]:
# Choose a file
#mat_files = 'lp_kb2.mat'     # b = 0
#mat_files = 'lp_afiro.mat'   # bmax = 500, and no mu tends to zero  and no condition problem
#mat_files = 'lp_blend.mat'    # bmax = 26.32, and no mu tends to zero.
mat_files = 'lp_fit1d.mat'   # b = 0
#mat_files = 'lp_fit1p.mat'   # bmax = 216, and no mu tends to zero, condition 10^12
#mat_files = 'lp_grow15.mat'  # b = 0, condition 10^6
#mat_files = 'lp_grow22.mat'  # b = 0, condition 1.4e7
#mat_files = 'lp_grow7.mat'   # b = 0, condition 1.2e7

# Load problem
Q, c, A, b, F, d, H = load_lp_problem(mat_files)

k = 0
n = Q.shape[0]
#print("n= ",n, "Q.shape[0]")
m = A.shape[0]
#print("m= ",m, "A.shape[0]")
p = F.shape[0]
#print("p= ",p, "F.shape[0]")

problem_info = {
    "problem_name": mat_files,
    "norm_inf_b": np.linalg.norm(b, np.inf),
    "dim_b": len(b),
    "num_zeros_b": np.sum(b==0),
    "dim_x": n,
    "dim_eq": m,
    "dim_ineq": p
}

tol = 1e-6
kmax = 100

#print("El rango de A es", np.linalg.matrix_rank(A))

AT = A.T
FT = F.T

# Initial values
x = np.ones(n)
lamda = np.zeros(m)
mu = np.ones(p)
z = np.ones(p)
#z = F @ x - d + (0.5)*np.ones(p)

sigma = 0.5 # Valor fijo
tau = sigma * np.dot(mu, z) / p

# Dataframes that store mu and z values on every iteration.
# Each iteration is a new row added after this.
# Used for the graphs
mudf = pd.DataFrame(columns=range(p))
zdf = pd.DataFrame(columns=range(p))
taudf = pd.DataFrame(columns=range(p))
ratiodf = pd.DataFrame(columns=range(p))
highlighted_df = pd.DataFrame(columns=range(p))
objdf = pd.DataFrame(columns=['obj'])
cmpdf = pd.DataFrame(columns=['cmp'])

#Dataframes which record mu, z, tau and objective function values over each iteration
mudf.loc[k] = mu
zdf.loc[k] = z
taudf.loc[k] = np.full(p, tau)
ratiodf.loc[k] = mu / tau
obj_val = 0.5 * x @ Q @ x + c @ x
cmp_val = np.max(mu * z)
objdf.loc[k] = obj_val
cmpdf.loc[k] = cmp_val

v1 = Q @ x + AT @ lamda - FT @ mu + c
v2 = A @ x - b
v3 = -F @ x + d + z
v4 = np.multiply(mu, z)  # Element-wise product
ld1 = np.concatenate((v1, v2, v3, v4), 0)
norma_cnpo = np.linalg.norm(ld1,np.inf) # this is the CNPO without the perturbation
w = np.zeros((p, 1))

# Initialize an empty DataFrame to store the iteration results
highlighted_df = pd.DataFrame(columns=range(p))  # drop/test inequalities → p columns
regression_df = pd.DataFrame(columns=highlighted_df.columns) #store any regressed indexes for mu

red_mu = []

while norma_cnpo > tol and k < kmax:
    # Update diagonal matrices Z and U inside the loop
    # Initial residuals
    Z = np.diag(z)
    U = np.diag(mu)
    ### KKT Matrix
    row1 = np.hstack((Q, AT, -FT, np.zeros((n, p))))
    row2 = np.hstack((A, np.zeros((m, m + p + p))))
    row3 = np.hstack((-F, np.zeros((p, m + p)), np.identity(p)))
    row4 = np.hstack((np.zeros((p, n + m)), Z, U))

    M = np.vstack((row1, row2, row3, row4))
    
    D = np.diag(mu / z)
    G = Q+FT@D@F
    for i in range(p):
        w[i] = F[i, :] @ x - d[i] - (tau / mu[i])
    w = w.ravel()
        
    dg = Q @ x + AT @ lamda - FT@mu + c + FT@D@w
    
    # Define K as a block matrix
    m = A.shape[0]
    K = np.block([
        [G, AT],
        [A, np.zeros((m, m))]
    ])
    
    # Calculate the reciprocal condition number of G
    condG = np.linalg.cond(G,1)
    
    # Define ld
    ld = -np.concatenate([dg, A @ x - b])
    #norma_cnpo = np.linalg.norm(ld, np.inf)
    
    # Solve the linear system and catch errors

    w_vector = solve_catch(K,ld) # Reduced system
    
    # Update the sections of the w_vector
    wx     = w_vector[0:n]
    wlamda = w_vector[n:n + m]
    wmu    = - D @ (F @ wx + w)
    wz     = - ( (1 / mu) * (z * wmu - tau) + z )
    
    ### Step size
    alpha_mu = paso_intpoint(mu, wmu)
    alpha_z  = paso_intpoint(z, wz)
    #alpha    = 0.995 * min(alpha_mu, alpha_z) # From Javi's thesis
    alpha    = min(alpha_mu, alpha_z)
    #print("alpha= " , alpha)
    
    # Percentage changes
    perc_mu = wmu/mu
    perc_z  = wz/z
    
    # Update variables
    x += alpha * wx
    mu += alpha * wmu
    lamda += alpha * wlamda
    z += alpha * wz
    
    # Update tau and residuals
    tau = sigma * np.dot(mu, z) / (p)
    k += 1

    mudf.loc[k] = mu    # dataframe for graphing the central path
    zdf.loc[k] = z  
    taudf.loc[k] = np.full(p, tau)     # dataframe for graphing the central path
    ratiodf.loc[k] = mu / tau
    obj_val = 0.5 * x @ Q @ x + c @ x
    cmp_val = np.max(mu * z)
    objdf.loc[k] = obj_val
    cmpdf.loc[k] = cmp_val

    highlighted_df = highlight_df(mu, z, Q, k, tau, highlighted_df, mudf)

    # Recalculate residuals
    v1 = Q @ x + AT @ lamda - FT @ mu + c
    v2 = A @ x - b
    v3 = -F @ x + d + z
    v4 = np.multiply(mu, z)  # Element-wise product
    
    ld1 = np.concatenate((v1, v2, v3, v4), 0)
    norma_cnpo = np.linalg.norm(ld1,np.inf)

    main_norm = norma_cnpo # CHECK IF THIS IS CORRECT HERE
    pr_before   = np.linalg.norm(v2, np.inf) # before name is misleading
    inq_before  = np.linalg.norm(v3, np.inf) # before name is misleading
    cmp_before  = np.max(v4) # before name is misleading
    tau_before  = tau # before name is misleading
    cond_before = condG # before name is misleading
    obj_before  =  obj_val

    #print("\niter=", k, "\t", "||cnpo||=", norma_cnpo)
    #print("Condition number of G:", np.linalg.cond(G,1))
    #print("rcond(G)", (1/np.linalg.cond(G,1)))
    #print("tau =",tau)
    #print(z)
    #print(mu)
    
    mask = mu*z <= 1e-5
    
    #print('cuantos chicos mu*z = %g, vector\n' % (sum(mask)), (mu*z)[mask])

    if all(mask): #Return True only if every entry in mask is True
        # Once the mu and z have gotten sufficiently small,
        # we record where we were before we start with the heuristic/experiment:

        #neg_mu_mask = (-0.52 < perc_mu) & (perc_mu < -0.48)
        #const_z_mask = (-0.01 < perc_z) & (perc_z < 0.01)
        grow_z_mask = (perc_z > -0.03) #& (perc_z < 0.01)
        neg_mu = np.arange( len(mask) )[grow_z_mask]
        
        highlighted_df = highlight_df(mu, z, Q, k, tau, highlighted_df, mudf)
        
        #print('mus chicos: vector\n', mu[neg_mu])
        if set(red_mu).issubset( neg_mu ):
            print ('IS subset: GOOD')
        else:
            print ('FAILS subset condition: BAD')
            
        #print('mus chicos: vector\n', neg_mu)
        #print('  change in percentages for mu \n', perc_mu[neg_mu] )
        #print('zs tending to positive contants\n', z[neg_mu] )
        #print('  Largest and smallest change for percentages in entries of z  \n', min(perc_z[neg_mu]), max(perc_z[neg_mu] ))
        red_mu = neg_mu.copy()
    
    # So here is when we start getting close to the issues of the matrix, as we already exited the while loop

print("Número de iteraciones hasta el criterio de paro:",k)
k_red = 0
#tol_red = 1e-10 #check this
max_red = 5
total_iter= k + k_red
highlighted_columns, regressed_columns = display_highlights(highlighted_df,regression_df)
regressed_df = pd.DataFrame(columns=['Iteration', 'Regressed_columns'])

#Build system outside of the loop:
Z = np.diag(z)
U = np.diag(mu)

    # ────────── Build full system ──────────
r1 = np.hstack((Q, AT, -FT @ U))
r2 = np.hstack((A, np.zeros((m, m + p))))
r3 = np.hstack((-U @ F, np.zeros((p, m)), -Z @ U))

M = np.vstack((r1, r2, r3))   # full system
cond_full = np.linalg.cond(M, 1)
    

#For recording the results with the new mu values once some turn zero
mudf0 = pd.DataFrame(columns=range(p))
cmpdf0 = pd.DataFrame(columns=['cmp'])
taudf0 = pd.DataFrame(columns=range(p))
ratiodf0 = pd.DataFrame(columns=range(p))

mu0 = mu.copy()
mu0[highlighted_columns] = 0
mudf0.loc[total_iter] = mu0

red_results_df = pd.DataFrame(columns=['Función Objetivo', 'max(mu*z)', 'dim(M1)', 'Condición de G (full)', 'Condición de M1 (sistema nuevo)', 'Filas/Columnas eliminadas comparadas al original'])
red_results_df.loc[total_iter] = [obj_val, cmp_val, M.shape[0], cond_full, cond_full, len(highlighted_columns)]

while k_red < max_red:
#while np.linalg.norm(ld1, np.inf) > tol and k_red < max_red:
    Z = np.diag(z)
    U = np.diag(mu)

        # ────────── Build full system ──────────
    r1 = np.hstack((Q, AT, -FT @ U))
    r2 = np.hstack((A, np.zeros((m, m + p))))
    r3 = np.hstack((-U @ F, np.zeros((p, m)), -Z @ U))

    M = np.vstack((r1, r2, r3))   # full system

    M, M1, U1, ld1 = build_reduced_system(Q, AT, FT, U, A, F, Z, mu, x, lamda, c, b, d, tau, highlighted_columns)

    cond_full = np.linalg.cond(M, 1)    # for full system
    cond_reduced = np.linalg.cond(M1, 1)  # for reduced system

    print("cond(M1, 1-norm):", np.linalg.cond(M1, 1))

    # SOLVE THE NEW SYSTEM
    w_vector1 = scipy.linalg.solve(M1, ld1)

    wx1     = w_vector1[:n] # should be the same ?
    wlamda1 = w_vector1[n:n + m] # should be the same ?
    Dwmu1    = w_vector1[n + m:]
    
    wmu1 = U1 @ Dwmu1  # Divide each element of wmu1 by the corresponding diagonal element of U

    # Here's the vector transformed back into the original size problem
    full_wmu = np.zeros_like(mu)
    active_indices = [i for i in range(p) if i not in highlighted_columns]
    for i, idx in enumerate(active_indices):
        full_wmu[idx] = wmu1[i]

    # ────────── complete here ──────────
    # 1. Reconstruct Δz from the 3rd KKT row (primal feasibility)
    # δz = F δx - (Fx - d - z)  == F @ wx1 - v3  (since v3 = -F@x + d + z)
    full_wz = F @ wx1 - (F @ x - d - z)

    # Recompute step sizes for the reduced-step directions **DOUBLE CHECK THIS LATER
    alpha_mu = paso_intpoint(mu, full_wmu)   # ensures μ + α·Δμ ≥ (1-τ)μ > 0
    alpha_z  = paso_intpoint(z,  full_wz)    # ensures z + α·Δz ≥ (1-τ)z > 0
    alpha    = min(alpha_mu, alpha_z)

    # Compute perc_mu, perc_z for the reduced step
    perc_mu = full_wmu / mu
    perc_z  = full_wz / z

    # 2. Apply the step with the same step‑size alpha
    x      += alpha * wx1
    lamda  += alpha * wlamda1
    mu     += alpha * full_wmu
    z      += alpha * full_wz

    k_red += 1
    total_iter= k + k_red

    # Update highlighted_df after this reduced iteration
    highlighted_df = highlight_df(mu, z, Q, total_iter, tau, highlighted_df, mudf)
    # Recompute the highlighted columns for the next reduced iteration
    highlighted_columns, regressed_columns = display_highlights(highlighted_df,regression_df)

    mu0 = mu.copy()
    mu0[highlighted_columns] = 0
    mudf0.loc[total_iter] = mu0

    # 4. Refresh τ and bump iteration counter (optional, for bookkeeping)
    tau = sigma * np.dot(mu, z) / p
    tau0 = sigma * np.dot(mu0, z) / p

    # 3. Recompute residuals for diagnostics
    v1 = Q @ x + AT @ lamda - FT @ mu + c
    v10 = Q @ x + AT @ lamda - FT @ mu0 + c     # dual residual
    v2 = A @ x - b                             # primal residual
    v3 = -F @ x + d + z                        # inequality residual
    v4 = mu * z 
    v40 = mu0 * z                               # complementarity product
    ld1 = np.concatenate((v1, v2, v3, v4))
    ld10 = np.concatenate((v10, v2, v3, v40))

    norm_after = np.linalg.norm(ld1, np.inf)
    norm_after0 = np.linalg.norm(ld10, np.inf)

    mudf.loc[total_iter] = mu    # dataframe for graphing the central path
    zdf.loc[total_iter] = z  
    taudf.loc[total_iter] = np.full(p, tau)     # dataframe for graphing the central path
    ratiodf.loc[total_iter] = mu / tau
    
    obj_val = 0.5 * x @ Q @ x + c @ x
    cmp_val = np.max(mu * z)
    cmp_val0 = np.max(mu0 * z)

    objdf.loc[total_iter] = obj_val
    cmpdf.loc[total_iter] = cmp_val

    cmpdf0.loc[total_iter] = cmp_val0
    taudf0.loc[total_iter] = np.full(p, tau0)     # dataframe for graphing the central path

        # --- CHECK IF ALL μ ARE ZERO ---
    if np.all(mu0 == 0):
        print(f"All μ are zero at iteration {total_iter}, skipping μ updates and ratio calculation")
        print(f"Final iteration reached at k_red={k_red}, all μ are zero.")
        tau0 = 0
        full_wmu = np.zeros_like(mu)
        perc_mu = np.zeros_like(mu)
        ratiodf0.loc[total_iter] = np.zeros_like(mu)  # avoids divide-by-zero warning
        k_red = max_red  # forces loop exit after this iteration
    else:
        ratiodf0.loc[total_iter] = mu0 / tau0

    # After the reduced-loop iterations finish, right before computing metrics_after:
    D = np.diag(mu / z)      # updated D for the new mu and z
    D0 = np.diag(mu0 / z)      # updated D with mu0 and z 
    G = Q + FT @ D @ F 
    G0 = Q + FT @ D0 @ F       # updated G
    condG = np.linalg.cond(G, 1)
    condG0 = np.linalg.cond(G0, 1)

    red_results_df.loc[total_iter] = [obj_val,cmp_val0,(M1.shape[0]),cond_full,np.linalg.cond(M1,1),(n+m+p-M1.shape[0])]
    #red_results_df0.loc[total_iter] = [obj_val,cmp_val0,(M1.shape[0]),cond_full,np.linalg.cond(M1,1),(n+m+p-M1.shape[0])]

    # AFTER computing highlighted_columns, regressed_columns in the reduced loop:

    if len(regressed_columns) > 0:
        regressed_df = pd.concat([regressed_df,
                                pd.DataFrame({'Iteration': [k_red],
                                                'Regressed_columns': [regressed_columns]})],
                                ignore_index=True)
        print(f"Iteration of reduced loop {k_red}: Regressed columns:", regressed_columns)
    else:
        print(f"Iteration of reduced loop  {k_red}: No regressed columns")


metrics_before = {
    "overall ||ld||∞": main_norm,
    "primal ||·||∞": pr_before,
    "ineq ||·||∞": inq_before,
    "max(mu*z)": cmp_before,
    "tau": tau_before,
    "cond(G)": cond_before,
    "objective": obj_before
}

metrics_after = {
    "overall ||ld||∞": np.linalg.norm(ld10, np.inf),
    "primal ||·||∞": np.linalg.norm(v2, np.inf),
    "ineq ||·||∞": np.linalg.norm(v3, np.inf),
    "max(mu*z)": np.max(v40),
    "tau": tau0,
    "cond(G)": condG0,  # updated condition number
    "objective": obj_val
}

summary_df = progress_summary_df_clean_simple(metrics_before, metrics_after)

display(summary_df)

summary_df = summary_excel(summary_df, mat_files, n, m, p, b, regressed_df)

print("Condition number of G (1-norm):", condG0)
print("Objective after:", obj_val)

Loading problem from: lp_fit1d.mat
 Norma infinita de b:  0.0
Problem loaded. n=1049, m=24, p=1049
Number of zeros in b: 24 (100.00%)
IS subset: GOOD
IS subset: GOOD
IS subset: GOOD
IS subset: GOOD
IS subset: GOOD
Número de iteraciones hasta el criterio de paro: 38
Consistently highlighted in last 3 iterations: [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 24, 25, 26, 28, 30, 31, 34, 36, 37, 40, 42, 43, 46, 47, 49, 51, 54, 56, 57, 58, 60, 61, 62, 64, 66, 68, 70, 72, 73, 74, 75, 76, 78, 79, 82, 84, 86, 88, 90, 92, 94, 96, 98, 99, 102, 103, 106, 108, 110, 112, 114, 116, 118, 119, 120, 121, 122, 124, 126, 128, 130, 131, 133, 134, 135, 138, 139, 141, 142, 143, 144, 145, 146, 147, 148, 149, 152, 154, 156, 158, 159, 160, 161, 162, 164, 166, 168, 170, 172, 173, 174, 175, 178, 179, 180, 181, 184, 186, 187, 188, 189, 191, 193, 194, 196, 198, 199, 202, 204, 206, 208, 209, 212, 213, 214, 216, 218, 220, 222, 224, 225, 228, 230, 232, 234, 236, 238, 240, 241, 244, 2

,Metric,Before,After,Improved?
0,overall ||ld||∞,5.829223e-07,1.363567e+03,False
1,primal ||·||∞,3.492460e-10,1.979060e-09,False
2,ineq ||·||∞,1.421085e-14,1.421085e-14,False
3,max(mu*z),5.829223e-07,0.000000e+00,True
4,tau,2.454087e-07,0.000000e+00,True
5,cond(G),1.894961e+12,1.000000e+00,True
6,objective,-9.025019e+04,-9.025019e+04,False


Progress summary with regressions saved for lp_fit1d.mat at progress_summary/progress_summary_lp_fit1d.mat.xlsx
Condition number of G (1-norm): 1.0
Objective after: -90250.18917414354


In [19]:
print(red_results_df)

# Plot objective vs iteration
plt.plot(obj_red.index, obj_red['obj'], marker='o')
plt.xlabel('Iteration')
plt.ylabel('Objective Function Value')
plt.title('Objective Function over Iterations')

# Format y-axis with 6 decimal places (no scientific notation)
plt.gca().yaxis.set_major_formatter(mtick.FormatStrFormatter('%.6f'))

plt.grid(True)
plt.show()

    Función Objetivo     max(mu*z)  dim(M1)  Condición de G (full)  \
38     -90250.189203  5.829223e-07   2122.0           4.719073e+11   
39     -90250.189174  5.200760e-07   1401.0           4.719073e+11   
40     -90250.189174  5.203748e-07   1391.0           4.719073e+11   
41     -90250.189174  0.000000e+00   1374.0           4.719073e+11   

    Condición de M1 (sistema nuevo)  \
38                     4.719073e+11   
39                     2.322945e+06   
40                     2.320539e+06   
41                     2.321834e+06   

    Filas/Columnas eliminadas comparadas al original  
38                                             721.0  
39                                             721.0  
40                                             731.0  
41                                             748.0  


NameError: name 'obj_red' is not defined

In [ ]:
import matplotlib.pyplot as plt

plt.plot(objdf.index, objdf['obj'], marker='o')
plt.xlabel('Iteration')
plt.ylabel('Objective Function Value')
plt.title('Objective Function over Iterations')
plt.grid(True)
plt.show()

In [18]:
red_results_df

,Función Objetivo,max(mu*z),dim(M1),Condición de G (full),Condición de M1 (sistema nuevo),Filas/Columnas eliminadas comparadas al original
38,-90250.189203,5.829223e-07,2122.0,4.719073e+11,4.719073e+11,721.0
39,-90250.189174,5.200760e-07,1401.0,4.719073e+11,2.322945e+06,721.0
40,-90250.189174,5.203748e-07,1391.0,4.719073e+11,2.320539e+06,731.0
41,-90250.189174,0.000000e+00,1374.0,4.719073e+11,2.321834e+06,748.0
